In [22]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [24]:
import os
print(os.listdir())

['.config', 'telugudataset.txt', 'merges.json', 'sample_data']


In [ ]:
with open("telugudataset.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:500])  # print first 500 chars

In [ ]:
print("Total characters:", len(text))

In [ ]:
# =========================
# 2. BUILD CHAR VOCAB
# =========================
chars = sorted(list(set(text)))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# convert text → ids
ids = [stoi[c] for c in text]

print("Initial IDs:", ids)
print(len(ids))

In [ ]:
def get_stats(ids):
    counts = {}

    for i in range(len(ids) - 1):
        pair = (ids[i], ids[i+1])
        counts[pair] = counts.get(pair, 0) + 1

    return counts


In [ ]:
def merge(ids, pair, idx):
    newids = []
    i = 0

    while i < len(ids):

        # if pair found → replace
        if i < len(ids)-1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2  # skip both elements

        else:
            newids.append(ids[i])
            i += 1  # move normally

    return newids

In [ ]:
num_merges = 6000
merges = {}

next_id = max(stoi.values()) + 1
ids_copy = ids[:]

for i in range(num_merges):
    stats = get_stats(ids_copy)

    if not stats:
        break

    # pick most frequent pair
    pair = max(stats, key=stats.get)

    # store rule
    merges[pair] = next_id

    # apply merge
    ids_copy = merge(ids_copy, pair, next_id)

    print(f"Merge {i+1}: {pair} -> {next_id}")

    next_id += 1

print("\nLearned merges:", merges)

In [ ]:
# =========================
# 6. ENCODE (APPLY RULES)
# =========================
def encode(text):
    ids = [stoi[c] for c in text]

    while True:
        stats = get_stats(ids)

        pair = None
        best_rank = float("inf")

        # find best valid merge
        for p in stats:
            if p in merges and merges[p] < best_rank:
                best_rank = merges[p]
                pair = p

        if pair is None:
            break

        ids = merge(ids, pair, merges[pair])

    return ids

In [ ]:
# =========================
# 7. BUILD VOCAB FOR DECODE
# =========================
vocab = {i: itos[i].encode('utf-8') for i in itos}

for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

In [ ]:
vocab_size = len(stoi) + len(merges)
print("Vocab size:", vocab_size)
print(vocab)

In [ ]:
# =========================
# 8. DECODE
# =========================
def decode(ids):
    tokens = b"".join(vocab[i] for i in ids)
    return tokens.decode("utf-8")

In [ ]:
# =========================
# 9. TEST
# =========================
encoded = encode("మాన్వి")
print("\nEncoded:", encoded)

decoded = decode(encoded)
print("Decoded:", decoded)

In [23]:
import json

merges_str = {str(k): v for k, v in merges.items()}

with open("merges.json", "w") as f:
    json.dump(merges_str, f)